# Natural Selector - Demo with Google.com

Query web elements using natural language with element-based embeddings.

**Requirements:**
```bash
pip install playwright sentence-transformers
playwright install chromium
```

In [1]:
import sys
import os
import asyncio

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from playwright.async_api import async_playwright
from natural_selector import Session
from natural_selector.integrations import SentenceTransformerEmbedder
from natural_selector.utils import capture_snapshot

print("✓ Imports ready")

✓ Imports ready


## 1. Capture Google.com with Playwright

In [2]:
# Capture page using Playwright CDP
snapshot = None

async def capture_google():
    global snapshot
    
    print("Launching browser...")
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()
        
        print("Navigating to google.com...")
        await page.goto('https://www.google.com', wait_until='networkidle')
        
        print("Waiting for page to stabilize...")
        await asyncio.sleep(3)
        
        print("Capturing CDP snapshot...")
        snapshot = await capture_snapshot(page)
        
        await browser.close()
        print("✓ Snapshot captured")

# Run the async function
await capture_google()

print(f"\nSnapshot info:")
print(f"  Documents: {len(snapshot.get('documents', []))}")
print(f"  Strings: {len(snapshot.get('strings', []))}")

Launching browser...
Navigating to google.com...
Waiting for page to stabilize...
Capturing CDP snapshot...
✓ Snapshot captured

Snapshot info:
  Documents: 1
  Strings: 808


## 2. Create Session and Page

In [3]:
from natural_selector.interfaces import LLM

# Dummy LLM for demo (or use OpenAILLM if you have API key)
class DummyLLM(LLM):
    def generate(self, query: str, context: str, system_prompt: str = "") -> str:
        return "button-1"  # Mock response

# Create session
session = Session(
    llm=DummyLLM(),
    embedder=SentenceTransformerEmbedder(),
    top_k=5
)

print("Creating page from CDP snapshot...")
page = session.create_page_from_cdp(snapshot)

print(f"✓ Page created: {page}")
print(f"  Total elements after filtering: {len(page._id_mapping)}")

/Users/stevewang/Github/web-auto/natural-selector/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Creating page from CDP snapshot...
✓ Page created: Page(elements=119)
  Total elements after filtering: 119


## 3. View Element IDs

In [4]:
# Show all element IDs
print("Element IDs in page (first 30):\n")
for i, element_id in enumerate(list(page._id_mapping.keys())[:30], 1):
    print(f"  {i:2d}. {element_id}")

Element IDs in page (first 30):

   1. body-1
   2. div-1
   3. div-2
   4. a-1
   5. a-2
   6. header-1
   7. div-3
   8. a-3
   9. a-4
  10. a-5
  11. a-6
  12. span-1
  13. svg-1
  14. div-4
  15. button-1
  16. svg-2
  17. form-1
  18. div-5
  19. div-6
  20. svg-3
  21. textarea-1
  22. div-7
  23. div-8
  24. svg-4
  25. div-9
  26. div-10
  27. svg-5
  28. div-11
  29. svg-6
  30. button-2


## 4. View Element Text Representations

This is what gets embedded for each element (tag, attributes, path, siblings, text).

In [5]:
from natural_selector._internal.page_index import PageIndex

# Show text representations for interesting elements
print("Sample Element Text Representations:\n")

# Find some interesting elements (buttons, inputs, links)
interesting = []
for element_id in page._id_mapping.keys():
    if any(tag in element_id for tag in ['button-', 'input-', 'a-', 'textarea-']):
        interesting.append(element_id)
        if len(interesting) >= 5:
            break

for element_id in interesting:
    node = page._id_mapping[element_id]
    text_repr = PageIndex._generate_element_repr(node, element_id, page._id_mapping)
    
    print(f"{'='*70}")
    print(f"Element: {element_id}")
    print(f"{'='*70}")
    print(text_repr)
    print()

Sample Element Text Representations:

Element: a-1
tag: a
attributes: href="https://about.google/?fg=1&utm_source=google-US&utm_medium=referral&utm_campaign=hp-header"
path: body-1 > div-1 > div-2 > a-1
text: "About"
next sibling: a-2 (text: "Store")

Element: a-2
tag: a
attributes: href="https://store.google.com/US?utm_source=hp_header&utm_medium=google_ooo&utm_campaign=GS100042&hl=en-US"
path: body-1 > div-1 > div-2 > a-2
text: "Store"
previous sibling: a-1 (text: "About")
next sibling: header-1 (text: "Gmail Images  Sign in")

Element: a-3
tag: a
attributes: aria-label="Gmail ", href="https://mail.google.com/mail/&ogbl"
path: body-1 > div-1 > div-2 > header-1 > div-3 > a-3
text: "Gmail"
next sibling: a-4 (text: "Images")

Element: a-4
tag: a
attributes: aria-label="Search for Images ", href="https://www.google.com/imghp?hl=en&ogbl"
path: body-1 > div-1 > div-2 > header-1 > div-3 > a-4
text: "Images"
previous sibling: a-3 (text: "Gmail")

Element: a-5
tag: a
attributes: aria-label="G

## 5. Build Index (Embed All Elements)

This creates vector embeddings for each element.

In [6]:
print("Building index (embedding all elements)...")
print("Note: First run downloads sentence-transformers model (~90MB)\n")

index = PageIndex.from_node_tree(
    root=page._root,
    id_mapping=page._id_mapping,
    embedder=session.embedder,
    llm=session.llm
)

print(f"✓ Indexed {len(index.elements)} elements")
print(f"  Embedding dimension: {len(index.elements[0].embedding) if index.elements else 'N/A'}")

Building index (embedding all elements)...
Note: First run downloads sentence-transformers model (~90MB)

✓ Indexed 119 elements
  Embedding dimension: 384


## 6. Test Vector Search (Without LLM)

Query elements using vector similarity search.

In [7]:
from natural_selector._internal.retriever import search_elements

# Test queries
queries = [
    "search button",
    "search input",
    "gmail link",
    "images link",
]

for query in queries:
    print(f"{'='*70}")
    print(f"Query: '{query}'")
    print(f"{'='*70}")
    
    results = search_elements(
        query_text=query,
        elements=index.elements,
        embedder=session.embedder,
        top_k=3
    )
    
    for i, (element, score) in enumerate(results, 1):
        print(f"\n  Result {i} (similarity: {score:.4f})")
        print(f"  Element ID: {element.element_id}")
        print(f"  Text representation:")
        # Show first 3 lines
        lines = element.text_repr.split('\n')[:3]
        for line in lines:
            print(f"    {line}")
        if len(element.text_repr.split('\n')) > 3:
            print(f"    ...")
    
    print()

Query: 'search button'

  Result 1 (similarity: 0.5745)
  Element ID: a-14
  Text representation:
    tag: a
    attributes: href="/advanced_search?hl=en&fg=1", role="menuitem"
    path: body-1 > div-1 > div-40 > div-41 > div-43 > g-popup-1 > g-menu-1 > a-14
    ...

  Result 2 (similarity: 0.5692)
  Element ID: input-3
  Text representation:
    tag: input
    attributes: value="Google Search", aria-label="Google Search", name="btnK", role="button", type="submit"
    path: body-1 > div-1 > div-4 > form-1 > div-5 > center-2 > input-3
    ...

  Result 3 (similarity: 0.5686)
  Element ID: input-1
  Text representation:
    tag: input
    attributes: value="Google Search", aria-label="Google Search", name="btnK", role="button", type="submit"
    path: body-1 > div-1 > div-4 > form-1 > div-5 > div-13 > div-17 > center-1 > input-1
    ...

Query: 'search input'

  Result 1 (similarity: 0.5743)
  Element ID: input-3
  Text representation:
    tag: input
    attributes: value="Google Search"

## 7. View LLM Context for a Query

See what context gets sent to the LLM.

In [8]:
query = "search button"

# Get top 3 elements
top_elements = search_elements(
    query_text=query,
    elements=index.elements,
    embedder=session.embedder,
    top_k=3
)

# Build LLM context (same as PageIndex.query() does)
context_parts = []
for i, (element, score) in enumerate(top_elements):
    context_parts.append(f"## Element {i+1} (relevance: {score:.2f})")
    context_parts.append(f"ID: {element.element_id}")
    context_parts.append(element.text_repr)
    context_parts.append("")

context = "\n".join(context_parts)

system_prompt = """You are a browser automation assistant. Given webpage elements and a user query, identify the relevant element ID.

The context shows element details with IDs like:
- button-1, div-2, input-3, etc.

IMPORTANT: Only return the element ID(s), nothing else.
- For single element: just "button-1"
- For multiple elements: "button-1, input-2, div-3"
- If not found: "NOT_FOUND"

Do not include explanations, just the ID."""

print("="*70)
print("COMPLETE LLM INPUT")
print("="*70)

print("\n" + "-"*70)
print("SYSTEM PROMPT:")
print("-"*70)
print(system_prompt)

print("\n" + "-"*70)
print("USER MESSAGE:")
print("-"*70)
user_message = f"Webpage Context:\n{context}\n\nQuery: {query}"
print(user_message)

print("\n" + "="*70)
print("END OF LLM INPUT")
print("="*70)

COMPLETE LLM INPUT

----------------------------------------------------------------------
SYSTEM PROMPT:
----------------------------------------------------------------------
You are a browser automation assistant. Given webpage elements and a user query, identify the relevant element ID.

The context shows element details with IDs like:
- button-1, div-2, input-3, etc.

IMPORTANT: Only return the element ID(s), nothing else.
- For single element: just "button-1"
- For multiple elements: "button-1, input-2, div-3"
- If not found: "NOT_FOUND"

Do not include explanations, just the ID.

----------------------------------------------------------------------
USER MESSAGE:
----------------------------------------------------------------------
Webpage Context:
## Element 1 (relevance: 0.57)
ID: a-14
tag: a
attributes: href="/advanced_search?hl=en&fg=1", role="menuitem"
path: body-1 > div-1 > div-40 > div-41 > div-43 > g-popup-1 > g-menu-1 > a-14
text: "Advanced search"
previous sibling: a-

## 8. Full Query with LLM (Optional)

**Requires OpenAI API key.** Uncomment to test.

In [9]:
# Uncomment to test with real LLM:

# from natural_selector.integrations import OpenAILLM

# session_llm = Session(
#     llm=OpenAILLM(api_key=os.getenv("OPENAI_API_KEY")),
#     embedder=SentenceTransformerEmbedder(),
#     top_k=5
# )

# page_llm = session_llm.create_page_from_cdp(snapshot)

# # Query with natural language
# queries = ["search button", "gmail link", "images link", "search input"]

# for query in queries:
#     print(f"\n{'='*70}")
#     print(f"Query: '{query}'")
#     print(f"{'='*70}")
#     
#     element = page_llm.select_one(query)
#     if element:
#         print(f"\n  Found: {element}")
#         print(f"  Tag: {element.tag}")
#         print(f"  Text: {element.text}")
#         print(f"  Attributes: {element.attributes}")
#         print(f"  XPath: {element.to_xpath()}")
#     else:
#         print("  Not found")

## Summary

**Element-Based Embeddings Pipeline:**

1. **Capture** → Playwright CDP snapshot from Google.com
2. **Parse** → `domnode` parses CDP into Node tree
3. **Filter** → Visibility + semantic filters (keeps only visible, interactive elements)
4. **Generate IDs** → Each element gets semantic ID (`button-1`, `input-2`)
5. **Create Representations** → Natural language text for each element:
   - Tag, attributes, full path
   - Previous/next siblings
   - Element text content
6. **Embed** → Vector embeddings for all elements
7. **Query** → 
   - Vector search retrieves top-k similar elements
   - LLM selects best match from context
   - Return element ID → Generate XPath/CSS

**Key Benefits:**
- 🎯 Precise element-level matching
- 🧭 Rich context (path + siblings)
- 🗣️ Natural language queries
- 🚫 No manual selectors needed
- 📊 Scales to large DOMs